### Load libraries

In [1]:
from pybaseball import statcast

import pandas as pd
import numpy as np
import polars as pl

import warnings
warnings.filterwarnings("ignore")

import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

import requests

import matplotlib.pyplot as plt
%matplotlib inline

from huggingface_hub import HfApi

In [2]:
# collect Statcast data
# documentation: https://baseballsavant.mlb.com/csv-docs
start_date = '2020-03-01'
end_date = '2025-10-31'

raw_data = statcast(start_date, end_date)
print(raw_data.shape)

This is a large query, it may take a moment to complete


  0%|                                                                                         | 0/1312 [00:00<?, ?it/s]

Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates
Skipping offseason dates


100%|██████████████████████████████████████████████████████████████████████████████| 1312/1312 [08:21<00:00,  2.62it/s]


(4126175, 118)


In [3]:
df = raw_data.copy().reset_index(drop=True)
# re-format names to readable format
split_names = df["player_name"].fillna("").str.split(",", n=1)
df["PlayerName"] = np.where(
    split_names.str.len() == 2,
    split_names.str[1].str.strip() + " " + split_names.str[0].str.strip(),
    df["player_name"]
)

In [4]:
## Define a dictionary to group outcomes together for expected run values
des_dict = {'ball':'ball',
'hit_into_play':'hit_into_play',
'called_strike':'called_strike',
'foul':'foul',
'swinging_strike':'swinging_strike',
'blocked_ball':'ball',
'swinging_strike_blocked':'swinging_strike',
'foul_tip':'swinging_strike',
'foul_bunt':'foul',
'hit_by_pitch':'hit_by_pitch',
'pitchout':'ball',
'missed_bunt':'swinging_strike',
'bunt_foul_tip':'swinging_strike',
'foul_pitchout':'foul',}

## Define a dictionary to group events together for expected run values
ev_dict = {'game_advisory':np.nan,
 'single':'single',
 'walk':'walk',
 np.nan:np.nan,
 'strikeout':'strikeout',
 'field_out':'field_out',
 'force_out':'field_out',
 'double':'double',
 'hit_by_pitch':'hit_by_pitch',
 'home_run':'home_run',
 'grounded_into_double_play':'field_out',
 'fielders_choice_out':'field_out',
 'fielders_choice':'field_out',
 'field_error':np.nan,
 'double_play':'field_out',
 'sac_fly':'field_out',
 'strikeout_double_play':np.nan,
 'triple':'triple',
 'caught_stealing_2b':np.nan,
 'sac_bunt':'field_out',
 'catcher_interf':np.nan,
 'caught_stealing_3b':np.nan,
 'sac_fly_double_play':'field_out',
 'triple_play':'field_out',
 'other_out':'field_out',
 'pickoff_3b':np.nan,
 'caught_stealing_home':np.nan,
 'pickoff_1b':np.nan,
 'pickoff_2b':np.nan,
 'wild_pitch':'wild_pitch',
 'stolen_base_2b':np.nan,
 'pickoff_caught_stealing_3b':np.nan,
 'pickoff_caught_stealing_2b':np.nan,
 'sac_bunt_double_play':np.nan,
 'passed_ball':np.nan,
 'pickoff_caught_stealing_home':np.nan,}

In [5]:
# Define a function which applies relevant transformations
# Code derived from Thomas Nestico

def clean_for_xrv(df, val_year):
    df = df[(df.balls != 4) & (df.strikes != 3)].copy()

    df["des_new"] = df["description"].map(des_dict)
    df["des_new"] = np.where((df["des_new"] == "ball") & (df["balls"] == 3), "walk", df["des_new"])
    df["des_new"] = np.where(
        (df["des_new"].isin(["called_strike", "swinging_strike"])) & (df["strikes"] == 2),
        "strikeout",
        df["des_new"],
    )
    df["ev_new"] = df.loc[df["des_new"] == "hit_into_play", "events"].map(ev_dict)
    df.loc[df["des_new"] == "hit_into_play", "des_new"] = df.loc[df["des_new"] == "hit_into_play", "ev_new"]
    df = df.dropna(subset=["des_new"])

    train_mask = (df["game_year"].to_numpy() < val_year)

    des_values = (
        df.loc[train_mask]
          .groupby(["des_new", "strikes", "balls"])["delta_run_exp"]
          .mean()
          .reset_index(name="delta_run_exp_mean")
    )

    df = df.merge(des_values, on=["des_new", "strikes", "balls"], how="left")

    global_mean = df.loc[train_mask, "delta_run_exp"].mean()
    df["delta_run_exp_mean"] = df["delta_run_exp_mean"].fillna(global_mean)

    return df.dropna(subset=["pitch_type"])

In [6]:
def add_primary_pitch_deltas(
    df,
    primary_choices=("FF", "SI", "FC"),
    *,
    pitcher_col="pitcher",
    year_col="game_year",
    pitch_type_col="pitch_type",
    speed_col="release_speed",
    pfx_x_col="pfx_x",
    pfx_z_col="pfx_z",
):
    """
    Adds primary pitch baseline per pitcher-season and deltas vs that baseline.

    Primary pitch definition:
      - among `primary_choices`, pick the most-used pitch type for a given pitcher-season
      - tiebreaker: higher avg speed

    Fallback if pitcher-season throws none of `primary_choices`:
      - use fastest pitch (by avg speed) for that pitcher-season

    Returns a pandas DataFrame with:
      - avg_primary_speed, avg_primary_pfx_x, avg_primary_pfx_z
      - primary_delta_release_speed, primary_delta_pfx_x, primary_delta_pfx_z
    """

    pdf = df.copy()
    pl_df = pl.from_pandas(pdf)

    group_keys = [pitcher_col, year_col, pitch_type_col]

    # aggregate pitch-type means and counts per pitcher-season
    agg = (
        pl_df.group_by(group_keys)
        .agg(
            pl.len().alias("pt_count"),
            pl.col(speed_col).mean().alias("pt_avg_speed"),
            pl.col(pfx_z_col).mean().alias("pt_avg_pfx_z"),
            pl.col(pfx_x_col).mean().alias("pt_avg_pfx_x"),
        )
    )

    # candidate primary among chosen pitch types
    primary_candidates = (
        agg.filter(pl.col(pitch_type_col).is_in(list(primary_choices)))
        .sort(
            by=[pitcher_col, year_col, "pt_count", "pt_avg_speed"],
            descending=[False, False, True, True],
        )
        .unique(subset=[pitcher_col, year_col], keep="first")
        .select(
            pitcher_col,
            year_col,
            pl.col("pt_avg_speed").alias("avg_primary_speed"),
            pl.col("pt_avg_pfx_z").alias("avg_primary_pfx_z"),
            pl.col("pt_avg_pfx_x").alias("avg_primary_pfx_x"),
        )
    )

    # fallback: fastest pitch type for that pitcher-season (if no primary_choices)
    fastest_fallback = (
        agg.sort(
            by=[pitcher_col, year_col, "pt_avg_speed"],
            descending=[False, False, True],
        )
        .unique(subset=[pitcher_col, year_col], keep="first")
        .select(
            pitcher_col,
            year_col,
            pl.col("pt_avg_speed").alias("fb_primary_speed"),
            pl.col("pt_avg_pfx_z").alias("fb_primary_pfx_z"),
            pl.col("pt_avg_pfx_x").alias("fb_primary_pfx_x"),
        )
    )

    # join both baselines to every pitch row
    pl_df = (
        pl_df.join(primary_candidates, on=[pitcher_col, year_col], how="left")
             .join(fastest_fallback, on=[pitcher_col, year_col], how="left")
             .with_columns(
                 pl.when(pl.col("avg_primary_speed").is_null())
                   .then(pl.col("fb_primary_speed"))
                   .otherwise(pl.col("avg_primary_speed"))
                   .alias("avg_primary_speed"),
                 pl.when(pl.col("avg_primary_pfx_z").is_null())
                   .then(pl.col("fb_primary_pfx_z"))
                   .otherwise(pl.col("avg_primary_pfx_z"))
                   .alias("avg_primary_pfx_z"),
                 pl.when(pl.col("avg_primary_pfx_x").is_null())
                   .then(pl.col("fb_primary_pfx_x"))
                   .otherwise(pl.col("avg_primary_pfx_x"))
                   .alias("avg_primary_pfx_x"),
             )
             .drop(["fb_primary_speed", "fb_primary_pfx_z", "fb_primary_pfx_x"])
    )

    # deltas vs baseline
    pl_df = pl_df.with_columns(
        (pl.col(speed_col) - pl.col("avg_primary_speed")).alias("primary_delta_release_speed"),
        (pl.col(pfx_z_col) - pl.col("avg_primary_pfx_z")).alias("primary_delta_pfx_z"),
        (pl.col(pfx_x_col) - pl.col("avg_primary_pfx_x")).alias("primary_delta_pfx_x"),
    )

    pl_df = pl_df.with_columns(pl.col(year_col).cast(pl.Int64))

    return pl_df.to_pandas()

In [7]:
def clean_data(df, val_year):
    
    df = df[df.game_type == 'R'].copy()  # regular season only

    # required columns
    must_have_cols = [
        'p_throws', 'pfx_x', 'pfx_z', 'release_speed', 'spin_axis',
        'release_spin_rate', 'release_extension',
        'release_pos_x', 'release_pos_z', 'arm_angle', 'p_throws'
    ]
    df = df.dropna(subset=must_have_cols)

    # remove unidentified pitches
    df = df[df.pitch_name != 'Other']

    # remove bunts
    bunt_plays = ['bunt_foul_tip', 'foul_bunt', 'missed_bunt']
    df = df[~df.description.isin(bunt_plays)]

    # remove clearly bad extensions
    df = df[df.release_extension <= 10]

    # rename batter stance
    df = df.rename(columns={'stand': 'batter_handedness'})

    # drop deprecated columns
    deprecated_columns = [c for c in df.columns if 'deprecated' in c]
    df = df.drop(columns=deprecated_columns)

    # filter rare pitch types globally
    vc = df['pitch_type'].value_counts()
    included_pitches = vc[vc >= 5000].index
    df = df[df.pitch_type.isin(included_pitches)]

    # handedness feature
    df['is_lefty'] = (df['p_throws'] == 'L').astype(int)

    # convert movement to inches
    df['pfx_x'] = df['pfx_x'] * 12.0
    df['pfx_z'] = df['pfx_z'] * 12.0
    
    # same_side matchup feature — captures platoon split context
    df['same_side'] = (df['p_throws'] == df['batter_handedness']).astype(int)

    # mirror pfx_x for model so arm-side is always positive regardless of handedness
    df['pfx_x_mirrored'] = df['pfx_x'] * np.where(df['p_throws'] == 'L', -1, 1)

    eps = 1e-9
    train_mask = df['game_year'] < val_year  # avoid leakage

    # -------------------------
    # spin efficiency
    # accounts for flight distance via extension; transverse spin back-calculated from observed movement
    # -------------------------
    SEAM_CONSTANT = 0.01267

    df['total_movement'] = np.sqrt(df['pfx_x']**2 + df['pfx_z']**2)
    df['flight_distance'] = 60.5 - df['release_extension']

    df['transverse_spin_est'] = (
        df['total_movement'] * df['release_speed']
        / (SEAM_CONSTANT * df['flight_distance'] + eps)
    )

    df['spin_efficiency'] = (
        df['transverse_spin_est'] / (df['release_spin_rate'] + eps)
    ).clip(0, 1)

    # -------------------------
    # spin axis features
    # -------------------------
    df['spin_axis'] = pd.to_numeric(df['spin_axis'], errors='coerce')
    df = df.dropna(subset=['spin_axis'])

    df['spin_axis_rad'] = np.deg2rad(df['spin_axis'])
    df['spin_axis_x'] = np.cos(df['spin_axis_rad'])
    df['spin_axis_y'] = np.sin(df['spin_axis_rad'])

    # -------------------------
    # seam-shifted wake features
    # Magnus magnitude = spin_efficiency fraction of observed movement
    # direction derived from spin axis
    # SSW = residual after removing Magnus estimate
    # convention: HB_obs > 0 -> 3B side, HB_obs < 0 -> 1B side
    # -------------------------
    df['HB_obs']  = -df['pfx_x']
    df['iVB_obs'] =  df['pfx_z']

    df['movement_mag_obs'] = np.sqrt(
        df['HB_obs']**2 + df['iVB_obs']**2
    )

    # Magnus magnitude = spin efficiency fraction of total movement
    df['magnus_mag_est'] = df['spin_efficiency'] * df['movement_mag_obs']

    # Magnus direction from spin axis
    theta = np.deg2rad(df['spin_axis'] - 90)
    df['HB_magnus_est']  = -df['magnus_mag_est'] * np.cos(theta)
    df['iVB_magnus_est'] =  df['magnus_mag_est'] * np.sin(theta)

    # SSW = residual after removing Magnus
    df['ssw_x'] = df['HB_obs'] - df['HB_magnus_est']
    df['ssw_z'] = df['iVB_obs'] - df['iVB_magnus_est']
    df['ssw_in'] = np.sqrt(df['ssw_x']**2 + df['ssw_z']**2)

    # Pitch usage filtering
    counts = df.groupby(['pitcher', 'game_year', 'pitch_type']).size().rename('count')
    totals = df.groupby(['pitcher', 'game_year']).size().rename('total')

    pitch_frequencies = (
        counts.to_frame()
              .join(totals, on=['pitcher', 'game_year'])
    )
    pitch_frequencies['usage'] = pitch_frequencies['count'] / pitch_frequencies['total']
    pitch_frequencies = pitch_frequencies.reset_index()

    pitch_frequencies = pitch_frequencies[
        (pitch_frequencies['count'] >= 10) &
        (pitch_frequencies['usage'] > 0.01)
    ]

    df = df.merge(
        pitch_frequencies[['pitcher', 'game_year', 'pitch_type']],
        on=['pitcher', 'game_year', 'pitch_type'],
        how='inner'
    )

    return df, pitch_frequencies

In [8]:
year_col = "game_year"
pitch_type_col = "pitch_type"
target = "delta_run_exp_mean"

all_years = sorted(df[year_col].dropna().astype(int).unique())
if len(all_years) < 3:
    raise ValueError(f"Need >=3 distinct years for train/tune/report. Found: {all_years}")

report_year = int(all_years[-1])
val_year    = int(all_years[-2])          # tune / early-stop year
train_years = [int(y) for y in all_years[:-2]]

print(f"Train years: {train_years}")
print(f"Tune year:   {val_year}")
print(f"Report year: {report_year}")

Train years: [2020, 2021, 2022, 2023]
Tune year:   2024
Report year: 2025


In [9]:
df = clean_for_xrv(df, val_year)

In [10]:
# Clean + engineer features
primary_choices = ['FF', 'SI', 'FC']
df, pitch_frequencies = clean_data(df, val_year)
df = add_primary_pitch_deltas(df, primary_choices=primary_choices)

In [11]:
ID_COLS = ["pitcher", "PlayerName"]

numeric_features = [
    'pfx_x_mirrored',
    'pfx_z',
    'release_speed', 'release_spin_rate', 'release_extension',
    'release_pos_x', 'release_pos_z',
    'spin_axis_x', 'spin_axis_y', 'spin_efficiency',
    'ssw_x', 'ssw_z', 'ssw_in',
    'same_side',
    'primary_delta_release_speed',
    'primary_delta_pfx_x', 'primary_delta_pfx_z',
]

categorical_features = [pitch_type_col]

model_features = numeric_features + categorical_features

KEEP_COLS = (
    ID_COLS
    + [year_col, target]
    + model_features
    + [
        "pfx_x",
        "spin_axis",
        "pitch_name",
        "p_throws",
        "batter_handedness",
        "arm_angle",
        "HB_obs",
        "iVB_obs",
        "movement_mag_obs",
        "magnus_mag_est",
        "HB_magnus_est",
        "iVB_magnus_est",
    ]
)

KEEP_COLS = list(dict.fromkeys(KEEP_COLS))

df_current = df[KEEP_COLS].copy()

In [12]:
def tune_refit_predict(
    df_all,
    tune_year,
    report_year,
    train_years,
    *,
    year_col="game_year",
    pitch_type_col="pitch_type",
    target_col=target,
    model_features=model_features,
    n_trials=75,
):

    n_estimators_max = 5000
    early_stopping_rounds = 100
    seed = 162

    # 1) Tune on tune_year
    df_train = df_all[df_all[year_col].isin(train_years)].copy()
    df_tune  = df_all[df_all[year_col] == tune_year].copy()

    X_train = df_train[model_features].copy()
    y_train = df_train[target_col].copy()
    X_tune  = df_tune[model_features].copy()
    y_tune  = df_tune[target_col].copy()

    X_train[pitch_type_col] = X_train[pitch_type_col].astype("category")
    X_tune[pitch_type_col]  = X_tune[pitch_type_col].astype("category")

    def objective(trial):
        params = {
            "random_state": seed,
            "metric": "rmse",
            "n_estimators": n_estimators_max,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.10, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 200, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 25.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 25.0, log=True),
            "force_row_wise": True,
            "verbosity": -1,
        }

        model = LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_tune, y_tune)],
            eval_metric="rmse",
            categorical_feature=[pitch_type_col],
            callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
        )
        preds = model.predict(X_tune)
        return mean_squared_error(y_tune, preds, squared=False)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # capture best params
    best_params = study.best_params.copy()
    best_params.update({
        "random_state": seed,
        "metric": "rmse",
        "force_row_wise": True,
        "verbosity": -1,
    })

    # fit a probe model once more to get best_iteration_
    probe_params = best_params.copy()
    probe_params["n_estimators"] = n_estimators_max

    probe_model = LGBMRegressor(**probe_params)
    probe_model.fit(
        X_train, y_train,
        eval_set=[(X_tune, y_tune)],
        eval_metric="rmse",
        categorical_feature=[pitch_type_col],
        callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
    )

    best_n_estimators = int(getattr(probe_model, "best_iteration_", None) or n_estimators_max)
    best_params["n_estimators"] = best_n_estimators

    tune_preds = probe_model.predict(X_tune)
    tune_rmse = mean_squared_error(y_tune, tune_preds, squared=False)

    # 2) Refit on (train_years + tune_year), params fixed
    refit_years = list(train_years) + [tune_year]
    df_refit = df_all[df_all[year_col].isin(refit_years)].copy()

    X_refit = df_refit[model_features].copy()
    y_refit = df_refit[target_col].copy()

    X_refit[pitch_type_col] = X_refit[pitch_type_col].astype("category")

    final_model = LGBMRegressor(**best_params)
    final_model.fit(
        X_refit, y_refit,
        categorical_feature=[pitch_type_col],
    )

    # 3) Predict/report on report_year
    df_report = df_all[df_all[year_col] == report_year].copy()
    X_report  = df_report[model_features].copy()
    X_report[pitch_type_col] = X_report[pitch_type_col].astype("category")

    df_report["pred_target_oos"] = final_model.predict(X_report)

    report_rmse = None
    if target_col in df_report.columns and df_report[target_col].notna().any():
        report_rmse = mean_squared_error(
            df_report[target_col],
            df_report["pred_target_oos"],
            squared=False,
        )

    feature_importance = pd.DataFrame({
        "feature": model_features,
        "importance": final_model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    result = {
        "report_year": report_year,
        "tune_year": tune_year,
        "train_years": train_years,
        "refit_years": refit_years,

        "study": study,
        "best_params": best_params.copy(),
        "best_n_estimators": best_n_estimators,

        "probe_model": probe_model,
        "final_model": final_model,

        "df_train": df_train,
        "df_tune": df_tune,
        "df_refit": df_refit,
        "df_report": df_report,

        "X_train": X_train,
        "y_train": y_train,
        "X_tune": X_tune,
        "y_tune": y_tune,
        "X_refit": X_refit,
        "y_refit": y_refit,
        "X_report": X_report,

        "tune_preds": tune_preds,
        "tune_rmse": tune_rmse,
        "report_rmse": report_rmse,

        "feature_importance": feature_importance,
    }

    return result

In [13]:
def rolling_oos_predictions(
    df_all,
    *,
    year_col="game_year",
    min_train_years=2,
    n_trials=75,
):
    years = sorted(df_all[year_col].dropna().astype(int).unique())

    all_preds = []
    results_by_year = {}
    models_by_year = {}
    summaries = []

    # Need at least [train years] + [tune year] + [report year]
    for i in range(min_train_years + 1, len(years)):
        report_year = int(years[i])
        tune_year = int(years[i - 1])
        train_years = [int(y) for y in years[:i - 1]]

        print(f"\n=== report_year={report_year} | tune_year={tune_year} | train_years={train_years} ===")

        result = tune_refit_predict(
            df_all,
            tune_year=tune_year,
            report_year=report_year,
            train_years=train_years,
            n_trials=n_trials,
        )

        print(
            f"Best RMSE: {result['study'].best_value:.6f} "
            f"| best_n_estimators: {result['best_n_estimators']}"
        )

        results_by_year[report_year] = result
        models_by_year[report_year] = result["final_model"]
        all_preds.append(result["df_report"])

        summaries.append({
            "report_year": result["report_year"],
            "tune_year": result["tune_year"],
            "train_years": result["train_years"],
            "n_train_rows": len(result["df_train"]),
            "n_tune_rows": len(result["df_tune"]),
            "n_refit_rows": len(result["df_refit"]),
            "n_report_rows": len(result["df_report"]),
            "best_value_rmse": result["study"].best_value,
            "tune_rmse": result["tune_rmse"],
            "report_rmse": result["report_rmse"],
            "best_n_estimators": result["best_n_estimators"],
            "best_params": result["best_params"],
        })

    if not all_preds:
        raise ValueError("No rolling predictions were created. Check available years and min_train_years.")

    predictions = pd.concat(all_preds, ignore_index=True)
    summary_df = pd.DataFrame(summaries)

    last_report_year = max(results_by_year.keys())

    return {
        "predictions": predictions,
        "summary": summary_df,
        "results_by_year": results_by_year,
        "models_by_year": models_by_year,
        "last_model": results_by_year[last_report_year]["final_model"],
        "last_result": results_by_year[last_report_year],
        "last_report_year": last_report_year,
    }

In [ ]:
rolling_results = rolling_oos_predictions(df_current)

df_pred_all = rolling_results["predictions"]
summary_df = rolling_results["summary"]
last_model = rolling_results["last_model"]
last_result = rolling_results["last_result"]
models_by_year = rolling_results["models_by_year"]
results_by_year = rolling_results["results_by_year"]


=== report_year=2023 | tune_year=2022 | train_years=[2020, 2021] ===


[I 2026-03-18 20:02:25,344] A new study created in memory with name: no-name-a5e5f888-f1c1-4630-af62-2e88f9e7579a
[I 2026-03-18 20:02:28,960] Trial 0 finished with value: 0.21462116754438154 and parameters: {'learning_rate': 0.06667824908427823, 'max_depth': 7, 'num_leaves': 66, 'min_child_samples': 15, 'subsample': 0.6162571484067767, 'colsample_bytree': 0.640543033051254, 'reg_alpha': 0.050619108398078316, 'reg_lambda': 23.433851978143604}. Best is trial 0 with value: 0.21462116754438154.
[I 2026-03-18 20:02:33,027] Trial 1 finished with value: 0.21461624175083285 and parameters: {'learning_rate': 0.04518728024008142, 'max_depth': 4, 'num_leaves': 139, 'min_child_samples': 47, 'subsample': 0.8985072799681042, 'colsample_bytree': 0.7849182855410712, 'reg_alpha': 4.533295186804083, 'reg_lambda': 10.876154527446506}. Best is trial 1 with value: 0.21461624175083285.
[I 2026-03-18 20:02:52,918] Trial 2 finished with value: 0.21461270924353715 and parameters: {'learning_rate': 0.0051916359

[I 2026-03-18 20:06:22,211] Trial 21 finished with value: 0.21460247248103828 and parameters: {'learning_rate': 0.025702333541613935, 'max_depth': 10, 'num_leaves': 97, 'min_child_samples': 10, 'subsample': 0.7712478390662965, 'colsample_bytree': 0.6845528775915285, 'reg_alpha': 18.733844184091332, 'reg_lambda': 0.08016224376702864}. Best is trial 16 with value: 0.2146019012384123.
[I 2026-03-18 20:06:28,002] Trial 22 finished with value: 0.2146109298972184 and parameters: {'learning_rate': 0.023251142814003967, 'max_depth': 10, 'num_leaves': 95, 'min_child_samples': 11, 'subsample': 0.7649263637264215, 'colsample_bytree': 0.9837489982972322, 'reg_alpha': 5.844262379549729, 'reg_lambda': 0.04403891100874083}. Best is trial 16 with value: 0.2146019012384123.
[I 2026-03-18 20:06:37,178] Trial 23 finished with value: 0.21460176602625555 and parameters: {'learning_rate': 0.014357945460429117, 'max_depth': 10, 'num_leaves': 71, 'min_child_samples': 25, 'subsample': 0.7178821363232363, 'cols

[I 2026-03-18 20:09:44,937] Trial 42 finished with value: 0.21460342003386507 and parameters: {'learning_rate': 0.016015293860306812, 'max_depth': 8, 'num_leaves': 69, 'min_child_samples': 49, 'subsample': 0.8533916523964877, 'colsample_bytree': 0.6307169591528903, 'reg_alpha': 7.743053211336315, 'reg_lambda': 0.9202182652272559}. Best is trial 31 with value: 0.21459903739753142.
[I 2026-03-18 20:09:53,064] Trial 43 finished with value: 0.21460459879893723 and parameters: {'learning_rate': 0.01777701348448384, 'max_depth': 9, 'num_leaves': 127, 'min_child_samples': 32, 'subsample': 0.8860855512509396, 'colsample_bytree': 0.594050248613703, 'reg_alpha': 3.2772657474253744, 'reg_lambda': 0.9384810744870401}. Best is trial 31 with value: 0.21459903739753142.
[I 2026-03-18 20:10:02,853] Trial 44 finished with value: 0.2146039100503056 and parameters: {'learning_rate': 0.014623621019683546, 'max_depth': 7, 'num_leaves': 52, 'min_child_samples': 25, 'subsample': 0.8089138492886878, 'colsampl

[I 2026-03-18 20:13:54,967] Trial 63 finished with value: 0.21460032690189143 and parameters: {'learning_rate': 0.008187920228072697, 'max_depth': 9, 'num_leaves': 49, 'min_child_samples': 125, 'subsample': 0.7944398258308378, 'colsample_bytree': 0.6612918158376534, 'reg_alpha': 14.134051780978488, 'reg_lambda': 0.4100074074497108}. Best is trial 31 with value: 0.21459903739753142.
[I 2026-03-18 20:14:13,657] Trial 64 finished with value: 0.21460076647154264 and parameters: {'learning_rate': 0.008946015478763121, 'max_depth': 9, 'num_leaves': 49, 'min_child_samples': 121, 'subsample': 0.7950576136413456, 'colsample_bytree': 0.6461128437133188, 'reg_alpha': 17.70929972458879, 'reg_lambda': 0.8301115442219582}. Best is trial 31 with value: 0.21459903739753142.
[I 2026-03-18 20:14:29,621] Trial 65 finished with value: 0.2146039394255624 and parameters: {'learning_rate': 0.008049690231314539, 'max_depth': 9, 'num_leaves': 26, 'min_child_samples': 132, 'subsample': 0.788825484566904, 'colsa

...

[I 2026-03-18 20:34:22,961] Trial 42 finished with value: 0.2194976355914853 and parameters: {'learning_rate': 0.0058342897122396105, 'max_depth': 7, 'num_leaves': 67, 'min_child_samples': 18, 'subsample': 0.8728458179470117, 'colsample_bytree': 0.9215244906233491, 'reg_alpha': 14.54738241787493, 'reg_lambda': 0.06877007543486216}. Best is trial 18 with value: 0.21949551193161043.
[I 2026-03-18 20:34:39,489] Trial 43 finished with value: 0.2195027898573524 and parameters: {'learning_rate': 0.01188946127443972, 'max_depth': 8, 'num_leaves': 53, 'min_child_samples': 25, 'subsample': 0.8270226201210168, 'colsample_bytree': 0.9970646091872609, 'reg_alpha': 8.073327641805745, 'reg_lambda': 0.0423014315241338}. Best is trial 18 with value: 0.21949551193161043.
[I 2026-03-18 20:35:10,787] Trial 44 finished with value: 0.2195044461232963 and parameters: {'learning_rate': 0.008457081077998373, 'max_depth': 7, 'num_leaves': 21, 'min_child_samples': 37, 'subsample': 0.7699843541697664, 'colsample

[I 2026-03-18 20:46:23,419] Trial 63 finished with value: 0.21949599834206376 and parameters: {'learning_rate': 0.009069927217680056, 'max_depth': 10, 'num_leaves': 135, 'min_child_samples': 68, 'subsample': 0.9431524940985428, 'colsample_bytree': 0.6272933767987707, 'reg_alpha': 13.716831933315975, 'reg_lambda': 0.23383633437385723}. Best is trial 18 with value: 0.21949551193161043.
[I 2026-03-18 20:46:54,879] Trial 64 finished with value: 0.21949665844589908 and parameters: {'learning_rate': 0.010949173151476975, 'max_depth': 10, 'num_leaves': 174, 'min_child_samples': 80, 'subsample': 0.9449050743986531, 'colsample_bytree': 0.6179651419377222, 'reg_alpha': 11.79538242003944, 'reg_lambda': 0.9965442710863085}. Best is trial 18 with value: 0.21949551193161043.
[I 2026-03-18 20:47:25,542] Trial 65 finished with value: 0.21949926178819829 and parameters: {'learning_rate': 0.011258735536644011, 'max_depth': 10, 'num_leaves': 180, 'min_child_samples': 106, 'subsample': 0.9105486529006926,

Best RMSE: 0.219495 | best_n_estimators: 226

=== report_year=2025 | tune_year=2024 | train_years=[2020, 2021, 2022, 2023] ===


[I 2026-03-18 20:51:25,063] A new study created in memory with name: no-name-721d5090-f6af-4cd9-9323-b47fbbf5a9a6
[I 2026-03-18 20:52:22,314] Trial 0 finished with value: 0.21635814269623127 and parameters: {'learning_rate': 0.011410288296799914, 'max_depth': 4, 'num_leaves': 165, 'min_child_samples': 24, 'subsample': 0.7967168038612205, 'colsample_bytree': 0.7170596523676519, 'reg_alpha': 0.2694806586012941, 'reg_lambda': 0.0028779326512478717}. Best is trial 0 with value: 0.21635814269623127.
[I 2026-03-18 20:52:34,920] Trial 1 finished with value: 0.21635903060537295 and parameters: {'learning_rate': 0.03761541177739543, 'max_depth': 7, 'num_leaves': 168, 'min_child_samples': 56, 'subsample': 0.6061097981422927, 'colsample_bytree': 0.8812240236205153, 'reg_alpha': 0.1037837728598877, 'reg_lambda': 0.001446566666950351}. Best is trial 0 with value: 0.21635814269623127.
[I 2026-03-18 20:54:10,090] Trial 2 finished with value: 0.21636577397367773 and parameters: {'learning_rate': 0.011

[I 2026-03-18 21:09:49,819] Trial 21 finished with value: 0.21634954446608262 and parameters: {'learning_rate': 0.007053943726913159, 'max_depth': 9, 'num_leaves': 40, 'min_child_samples': 78, 'subsample': 0.8364565179632212, 'colsample_bytree': 0.5540231080013295, 'reg_alpha': 7.885658984455039, 'reg_lambda': 9.527920049375185}. Best is trial 21 with value: 0.21634954446608262.
[I 2026-03-18 21:10:20,972] Trial 22 finished with value: 0.2163513340182604 and parameters: {'learning_rate': 0.01176376092729771, 'max_depth': 9, 'num_leaves': 42, 'min_child_samples': 119, 'subsample': 0.914863990961631, 'colsample_bytree': 0.5518993359809669, 'reg_alpha': 0.9717581493712939, 'reg_lambda': 11.163918426660715}. Best is trial 21 with value: 0.21634954446608262.
[I 2026-03-18 21:11:19,337] Trial 23 finished with value: 0.21634830227306007 and parameters: {'learning_rate': 0.007039112353895863, 'max_depth': 8, 'num_leaves': 79, 'min_child_samples': 70, 'subsample': 0.83489039524029, 'colsample_b

In [ ]:
stuff_sd = 10
eps = 1e-9

d = df_pred_all.copy()

# 1) Global Stuff+ — normalized within each year
# ensures 110 in 2022 means the same as 110 in 2025
mu_g = d.groupby(year_col)["pred_target_oos"].transform("mean")
sd_g = d.groupby(year_col)["pred_target_oos"].transform(
    lambda s: s.std(ddof=0)
)
z_g = (d["pred_target_oos"] - mu_g) / (sd_g + eps)

# Lower run value = better → higher Stuff+
d["Stuff+_global"] = 100 - stuff_sd * z_g

# 2) Pitch-type Stuff+ — normalized within each year and pitch type
pt_mu = d.groupby([year_col, pitch_type_col])["pred_target_oos"].transform("mean")
pt_sd = d.groupby([year_col, pitch_type_col])["pred_target_oos"].transform(
    lambda s: s.std(ddof=0)
)
z_pt = (d["pred_target_oos"] - pt_mu) / (pt_sd + eps)

d["Stuff+_pt"] = 100 - stuff_sd * z_pt

df_scored = d

In [ ]:
arsenal = (
    df_scored.groupby(["pitcher", "game_year", "pitch_type"])
      .size()
      .rename("count")
      .reset_index()
)

totals = df_scored.groupby(["pitcher", "game_year"]).size().rename("total").reset_index()

arsenal = arsenal.merge(totals, on=["pitcher", "game_year"])
arsenal["usage"] = arsenal["count"] / arsenal["total"]

# Define a pitch to be part of a pitcher's arsenal if it is thrown 3% of the time and at least 100 times overall
min_frequency = 0.03
min_count = 100
arsenal["is_arsenal"] = (arsenal["count"] >= min_count) & (arsenal["usage"] >= min_frequency)

df_scored = df_scored.merge(
    arsenal[["pitcher", "game_year", "pitch_type", "is_arsenal"]],
    on=["pitcher", "game_year", "pitch_type"],
    how="left"
)

In [ ]:
def fetch_fangraphs_pitching_stats(season: int, qual: int = 0, pageitems: int = 200000):
    
    base_url = "https://www.fangraphs.com/api/leaders/major-league/data"
    params = {
        "pos": "all",
        "stats": "pit",
        "lg": "all",
        "season": season,
        "season1": season,
        "ind": 1,
        "qual": qual,
        "type": 8,
        "month": 0,
        "pageitems": pageitems,
    }

    r = requests.get(base_url, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()

    df = pd.DataFrame(payload["data"])
    
    required = {"xMLBAMID", "IP"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"FanGraphs response missing columns: {missing}. Available: {sorted(df.columns)[:20]} ...")
    
    # ensure Season exists for filtering/merging consistency
    if "Season" not in df.columns:
        df["Season"] = season

    return df

In [ ]:
# Fetch IP for all scored years, not just report_year
scored_years = sorted(rolling_results["results_by_year"].keys())

ip_all_years = []
for yr in scored_years:
    stats = fetch_fangraphs_pitching_stats(season=yr, qual=0)
    stats = stats[stats["Season"] == yr][["xMLBAMID", "IP", "Season"]].copy()
    ip_all_years.append(stats)

ip_all = pd.concat(ip_all_years, ignore_index=True).rename(columns={"Season": "game_year"})

# Merge IP into df_scored for all years
df_scored = df_scored.merge(
    ip_all,
    left_on=["pitcher", "game_year"],
    right_on=["xMLBAMID", "game_year"],
    how="left"
).drop(columns=["xMLBAMID"])

assert "IP" in df_scored.columns, (
    "IP missing from df_scored — FanGraphs merge did not run. "
    "Re-run this cell before proceeding to export."
)

In [ ]:
def stuff_movement_heatmap(df, stuff_col="Stuff+_global"):
    
    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(
        df["pfx_x"],
        df["pfx_z"],
        C=df[stuff_col],
        reduce_C_function=np.mean,
        gridsize=30,
        cmap="RdBu_r",
        mincnt=1
    )
    
    x_lo, x_hi = df['pfx_x'].quantile([0.02, 0.98])
    y_lo, y_hi = df['pfx_z'].quantile([0.02, 0.98])

    ax.set_xlim(x_lo - 1.5, x_hi + 1.5)
    ax.set_ylim(y_lo - 1.5, y_hi + 1.5)

    ax.set_xlabel("Horizontal Break (inches)")
    ax.set_ylabel("Vertical Break (inches)")

    ax.axvline(0, linestyle="--", linewidth=1)
    ax.axhline(0, linestyle="--", linewidth=1)

    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Stuff+")

    plt.tight_layout()
    plt.show()

In [ ]:
df_p = df_scored[df_scored['PlayerName'] == 'Jacob deGrom']
df_p = df_p[df_p.pitch_type == 'FF']

stuff_movement_heatmap(df_p, stuff_col="Stuff+_global")

In [ ]:
pitcher_history = (
    df_scored
    .groupby(["game_year", "PlayerName", "pitch_type"], as_index=False)
    .agg(
        Pitches=("pitch_type", "size"),
        StuffPlus=("Stuff+_pt", "mean"),
        Velo=("release_speed", "mean"),
        HB=("HB_obs", "mean"),
        iVB=("pfx_z", "mean"),
        Spin=("release_spin_rate", "mean"),
        SpinAxis=("spin_axis", "mean"),
        SpinEff=("spin_efficiency", "mean"),
        SSW=("ssw_in", "mean"),
        ArmAngle=("arm_angle", "mean"),
        Extension=("release_extension", "mean"),
    )
)

# Add IP — one value per pitcher-season, broadcast across all pitch type rows
ip_per_pitcher = (
    df_scored[["PlayerName", "game_year", "IP"]]
    .dropna(subset=["IP"])
    .groupby(["PlayerName", "game_year"])["IP"]
    .max()
    .reset_index()
)

pitcher_history = pitcher_history.merge(
    ip_per_pitcher, on=["PlayerName", "game_year"], how="left"
)

min_year = min(rolling_results["results_by_year"].keys())
pitcher_history = pitcher_history[pitcher_history['game_year'] >= min_year].copy()
pitcher_history.to_parquet("pitcher_history.parquet", index=False)

In [ ]:
# -----------------------------
# Define export columns
# -----------------------------

EXPORT_COLS = [
    "pitcher",
    "PlayerName",
    "IP",
    "game_year",
    "pitch_type",
    "pitch_name",

    "p_throws",
    "batter_handedness",
    "is_lefty",

    "release_speed",
    "release_spin_rate",
    "spin_axis",
    "spin_axis_x",
    "spin_axis_y",
    "spin_efficiency",

    "pfx_x",
    "pfx_z",
    "HB_obs",
    "iVB_obs",

    "HB_magnus_est",
    "iVB_magnus_est",

    "ssw_x",
    "ssw_z",
    "ssw_in",

    "release_pos_x",
    "release_pos_z",
    "release_extension",
    "flight_distance",
    "arm_angle",

    "primary_delta_release_speed",
    "primary_delta_pfx_x",
    "primary_delta_pfx_z",

    "pred_target_oos",
    "Stuff+_global",
    "Stuff+_pt",

    "is_arsenal",
]

# -----------------------------
# Shrink dataframe for export
# -----------------------------

EXPORT_COLS_PRESENT = [c for c in EXPORT_COLS if c in df_scored.columns]

df_export = df_scored[EXPORT_COLS_PRESENT].copy()

In [ ]:
api = HfApi()
repo_id = "perld/stuff-plus-data"

for year, df_year in df_export.groupby("game_year"):
    filename = f"df_scored_{int(year)}.parquet"
    df_year.to_parquet(filename, index=False)
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="dataset",
    )

api.upload_file(
    path_or_fileobj="pitcher_history.parquet",
    path_in_repo="pitcher_history.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

In [ ]:
player_pitch = (
    df_scored[df_scored['is_arsenal']]
    .groupby(["pitcher", pitch_type_col])
    .agg(
        n_pitches=("pred_target_oos", "size"),
        StuffPlus_pt=("Stuff+_pt", "mean"),
        StuffPlus_global=("Stuff+_global", "mean"),
    )
    .reset_index()
)

pitcher_overall = (
    df_scored[df_scored['is_arsenal']]
    .groupby(["pitcher"])
    .agg(
        n_pitches=("pred_target_oos", "size"),
        StuffPlus_pt=("Stuff+_pt", "mean"),
        StuffPlus_global=("Stuff+_global", "mean"),
    )
    .reset_index()
)

In [ ]:
# Wide tables: one row per pitcher, pitch types as columns
wide_pt = player_pitch.pivot(index="pitcher", columns=pitch_type_col, values="StuffPlus_pt")
wide_global = player_pitch.pivot(index="pitcher", columns=pitch_type_col, values="StuffPlus_global")

# Combine into one table with suffixes
wide = (
    wide_pt.add_suffix("_pt")
    .join(wide_global.add_suffix("_global"), how="outer")
    .reset_index()
)

# Add pitcher-level overall Stuff+
wide = wide.merge(pitcher_overall, on="pitcher", how="left")

# Rename for readability
wide = wide.rename(columns={
    "StuffPlus_pt": "Stuff+_pt_overall",
    "StuffPlus_global": "Stuff+_global_overall",
})

pitcher_to_name = (
    df_scored[["pitcher", "PlayerName"]]
    .dropna(subset=["pitcher", "PlayerName"])
    .drop_duplicates(subset=["pitcher"])
    .set_index("pitcher")["PlayerName"]
)

wide["PlayerName"] = wide["pitcher"].map(pitcher_to_name)
cols = ["PlayerName", "pitcher"] + [c for c in wide.columns if c not in ["PlayerName", "pitcher"]]
wide = wide[cols]

# Merge IP into wide for the qualified leaderboard (report year only)
ip_report = ip_all[ip_all["game_year"] == report_year][["xMLBAMID", "IP"]].copy()
wide = wide.merge(
    ip_report,
    left_on="pitcher",
    right_on="xMLBAMID",
    how="left"
).drop(columns=["xMLBAMID"])

In [ ]:
qualified = wide[wide["IP"] >= 162].sort_values("Stuff+_global_overall", ascending=False)
qualified

In [ ]:
# pitch counts + usage (by pitcher-season)
counts = (
    df_scored.groupby(["pitcher", "PlayerName", 'game_year', 'pitch_type'])
    .size()
    .rename("n_pitches")
    .reset_index()
)

totals = (
    df_scored.groupby(["pitcher", 'game_year'])
    .size()
    .rename("n_total")
    .reset_index()
)

counts = counts.merge(totals, on=["pitcher", 'game_year'], how="left")
counts["usage"] = counts["n_pitches"] / counts["n_total"]

# pitch-level Stuff+ (restricted to arsenal pitches)
pitch_stuff = (
    df_scored[df_scored["is_arsenal"]]
    .groupby(["pitcher", "PlayerName", 'game_year', 'pitch_type'])
    .agg(
        StuffPlus_pt=("Stuff+_pt", "mean"),
        StuffPlus_global=("Stuff+_global", "mean"),
    )
    .reset_index()
)

# Join usage onto pitch-level stuff
arsenal_profile = (
    pitch_stuff.merge(counts, on=["pitcher", "PlayerName", 'game_year', 'pitch_type'], how="left")
    .sort_values(["PlayerName", 'game_year', "usage"], ascending=[True, True, False])
)

arsenal_profile.head()

In [ ]:
# Pitcher-season arsenal
tmp = arsenal_profile.copy()
tmp["w_pt"]     = tmp["usage"] * tmp["StuffPlus_pt"]
tmp["w_global"] = tmp["usage"] * tmp["StuffPlus_global"]

arsenal_summary = (
    tmp.groupby(["pitcher", "PlayerName", "game_year"])
    .agg(
        n_total=("n_total", "max"),
        n_arsenal_pitches=("pitch_type", "nunique"),

        StuffPlus_pt_overall=(
            "w_pt",
            lambda x: x.sum() / tmp.loc[x.index, "usage"].sum()
        ),
        StuffPlus_global_overall=(
            "w_global",
            lambda x: x.sum() / tmp.loc[x.index, "usage"].sum()
        ),
        top_pitch=(
            "StuffPlus_pt",
            lambda s: tmp.loc[s.index[s.argmax()], "pitch_type"]
        ),
        top_pitch_stuff=("StuffPlus_pt", "max"),
        primary_pitch=(
            "usage",
            lambda s: tmp.loc[s.index[s.argmax()], "pitch_type"]
        ),
        primary_pitch_usage=("usage", "max"),
    )
    .reset_index()
    .sort_values(["game_year", "StuffPlus_global_overall"], ascending=[False, False])
)

arsenal_summary.head()

In [ ]:
# Feature importances by year
importance_rows = []

for yr, result in rolling_results["results_by_year"].items():
    importances = result["final_model"].feature_importances_
    for feature, importance in zip(model_features, importances):
        importance_rows.append({
            "year": yr,
            "feature": feature,
            "importance": importance,
        })

importance_df = pd.DataFrame(importance_rows)

importance_wide = importance_df.pivot(
    index="feature",
    columns="year",
    values="importance"
)

# Add average across years
importance_wide["avg"] = importance_wide.mean(axis=1)

# Sort by average importance descending
importance_wide = importance_wide.sort_values("avg", ascending=False)

importance_wide.astype(int)